# H6a: LR Confound Audit of D-TEST -- The Paper's Credibility Depends on This

## Scientific Context and Motivation

The **D-TEST** (depth-exponent scaling test) produced one of the strongest quantitative claims in this research program: that Muon's advantage over SGD grows **exponentially with network depth**, following a per-layer multiplicative factor of ~1.10x with a linear fit $R^2 = 0.91$. Specifically, D-TEST measured:

$$\text{advantage}(L) = \frac{\text{best SGD loss at depth } L}{\text{best Muon loss at depth } L} \sim e^{0.0953 \cdot L}$$

This is a powerful claim. If true, it means Muon's spectral equalization via Newton-Schulz orthogonalization provides a **compounding** benefit: each additional layer amplifies the advantage. This would be strong evidence that Muon is not merely "a different optimizer" but addresses a **structural** limitation of SGD in deep networks -- namely, the exponential condition number growth of the end-to-end Jacobian $W_L \cdots W_1$.

### The Confound: Learning Rate Selection Asymmetry

However, **H6 revealed a critical flaw** in the original curvature rescaling experiments: a claimed 130x curvature advantage turned out to be entirely an LR artifact (the rescaler was just multiplying gradients by 0.1, equivalent to a 10x LR reduction). This raised an urgent question about D-TEST:

**D-TEST used asymmetric LR selection:**
- **SGD LR**: Tuned per depth using the formula $\text{lr} = 2 / (\lambda_{\max} \cdot L)$, where $\lambda_{\max}$ is the top eigenvalue of the Hessian proxy. This formula *decreases* the LR with depth.
- **Muon LR**: Fixed at 0.005 across all depths. No depth-dependent adjustment.

This asymmetry is suspicious. If the formula gives SGD an increasingly suboptimal LR at greater depths (too conservative or too aggressive), while Muon's fixed LR happens to be near-optimal across depths, then the measured "exponential scaling" could be an **LR mismatch artifact** rather than a genuine algorithmic advantage.

### What This Experiment Does

H6a applies a **fair LR audit** to D-TEST by sweeping learning rates for **both** optimizers at **every** depth:

| Depth $L$ | SGD | Muon |
|-----------|-----|------|
| 2, 4, 8, 16 | Sweep 9 LR candidates | Sweep 8 LR candidates |
| Each LR | 3 seeds, take median | 3 seeds, take median |
| Selection | Best LR by median loss | Best LR by median loss |

Then we measure advantage = best_SGD_loss / best_Muon_loss at each depth and fit $\log(\text{advantage})$ vs $L$.

### Verdict Criteria

| Outcome | Condition | Interpretation |
|---------|-----------|----------------|
| **D-TEST SURVIVES** | $R^2 > 0.8$, slope $> 0.03$ | Exponential scaling is real, not an LR artifact |
| **PARTIALLY SURVIVES** | $0.5 < R^2 < 0.8$ | Trend exists but is weaker after LR correction |
| **D-TEST FAILS** | $R^2 < 0.5$ or slope $< 0.01$ | Exponential scaling was an LR confound -- retract the claim |

### Why This Matters for the Muon-as-Gauge-Fixing Theory

The depth-exponent result is one of the pillars connecting Muon to renormalization group (RG) theory. In the RG interpretation, Muon's orthogonalization acts as a **gauge-fixing** operation that removes redundant spectral degrees of freedom at each layer. If this gauge-fixing is genuine, its benefit should **compound** with depth because each layer introduces new gauge redundancy. If the depth exponent disappears under fair LR tuning, then either (a) the RG connection is weaker than claimed, or (b) both optimizers can handle depth-related conditioning when given proper LRs, which would mean the advantage is about **LR sensitivity** rather than algorithmic structure.

In [ ]:
import numpy as np
import os
import time

print("=" * 90)
print("  H6a: LR CONFOUND AUDIT OF D-TEST DEPTH SCALING")
print("=" * 90)
print()
print(f"  NumPy version: {np.__version__}")
print(f"  Environment: Pure NumPy (no PyTorch) -- deep linear networks only")
print(f"  This ensures the experiment tests optimization dynamics,")
print(f"  not framework-specific numerical behavior.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"  Script directory: {SCRIPT_DIR}")

## Experimental Configuration

### Matching D-TEST Setup with Extended LR Sweeps

The architecture and training configuration replicate D-TEST exactly: 32x32 deep linear networks at depths {2, 4, 8, 16}, trained for 300 steps with momentum 0.9 and batch size 64. The only change is **how we select learning rates**: instead of using a formula for SGD and a fixed value for Muon, we sweep a broad range of candidates for both.

**Design rationale for parameters:**
- **DIM=32**: Large enough for nontrivial spectral structure (32 singular values per layer) but small enough for exhaustive sweeps. At depth 16, the product $W_{16} \cdots W_1$ is a 32x32 matrix with potentially extreme condition numbers.
- **DEPTHS=[2, 4, 8, 16]**: Geometric progression spanning an 8x range. If per-layer advantage is multiplicative, we expect $\log(\text{advantage})$ to be linear in $L$.
- **NUM_STEPS=300**: Sufficient for convergence in this architecture. We also measure advantage at intermediate steps (50, 100, ..., 300) to check if the depth scaling is transient or persistent.
- **NUM_SEEDS=3**: Modest but sufficient for median-based selection. We use median (not mean) to be robust to occasional divergence at aggressive LRs.
- **NS_ITERS=5**: Standard Newton-Schulz iterations for Muon. At 5 iterations, the polar factor approximation is accurate to machine precision for well-conditioned gradients.

In [ ]:
DIM = 32
DEPTHS = [2, 4, 8, 16]
NUM_STEPS = 300
MOMENTUM = 0.9
NS_ITERS = 5
BATCH_SIZE = 64
NUM_SEEDS = 3

print("=== Core Experimental Parameters ===")
print(f"  Network dimension:  {DIM}x{DIM} weight matrices")
print(f"  Depths to test:     {DEPTHS}")
print(f"  Training steps:     {NUM_STEPS}")
print(f"  Momentum:           {MOMENTUM}")
print(f"  Newton-Schulz iters: {NS_ITERS}")
print(f"  Batch size:         {BATCH_SIZE}")
print(f"  Seeds per config:   {NUM_SEEDS}")
print()
print(f"  At depth {DEPTHS[-1]}, the forward pass is a product of {DEPTHS[-1]} matrices.")
print(f"  Condition number of the product can grow as kappa^L,")
print(f"  making optimization exponentially harder with depth for SGD")
print(f"  but potentially not for Muon (if spectral equalization helps).")

In [ ]:
# LR sweep ranges — extended to avoid hitting sweep boundaries
SGD_LRS = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
MUON_LRS = [0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]

print("=== Learning Rate Sweep Design ===")
print(f"  SGD LR candidates  ({len(SGD_LRS)} values):  {SGD_LRS}")
print(f"  Muon LR candidates ({len(MUON_LRS)} values): {MUON_LRS}")
print()
print(f"  SGD range:  {SGD_LRS[0]} to {SGD_LRS[-1]} ({SGD_LRS[-1]/SGD_LRS[0]:.0f}x span)")
print(f"  Muon range: {MUON_LRS[0]} to {MUON_LRS[-1]} ({MUON_LRS[-1]/MUON_LRS[0]:.0f}x span)")
print()
print(f"  D-TEST used fixed Muon LR = 0.005 and formula SGD LR = 2/(lambda_max * L).")
print(f"  Here we sweep BOTH, so neither optimizer is handicapped by a bad LR choice.")
print()
total_runs = len(DEPTHS) * (len(SGD_LRS) + len(MUON_LRS)) * NUM_SEEDS
print(f"  Total training runs in the sweep: {total_runs}")
print(f"    = {len(DEPTHS)} depths x ({len(SGD_LRS)} SGD + {len(MUON_LRS)} Muon) LRs x {NUM_SEEDS} seeds")

## Network and Training Utilities

### Deep Linear Network Primitives

The deep linear network is defined by a chain of $L$ weight matrices $W_1, W_2, \ldots, W_L \in \mathbb{R}^{d \times d}$. The forward pass computes:

$$\hat{Y} = W_L \cdot W_{L-1} \cdots W_1 \cdot X$$

Despite having no nonlinearities, deep linear networks exhibit rich optimization dynamics that are central to this experiment:

1. **Exponential condition number growth**: If each $W_l$ has condition number $\kappa$, the product $W_L \cdots W_1$ can have condition number up to $\kappa^L$. SGD's convergence rate depends on this condition number, so deeper networks are exponentially harder for SGD.

2. **Muon's spectral equalization**: Newton-Schulz orthogonalization replaces the gradient $G$ with its polar factor $G_{\text{orth}} \approx UV^T$ (where $G = U\Sigma V^T$), setting all singular values of the update to 1. This potentially neutralizes the condition number's effect on optimization.

3. **Layer-dependent gradient scaling**: In a depth-$L$ network, the gradient with respect to layer $l$ involves products of $L-1$ other weight matrices. This creates a spectrum of gradient magnitudes across layers, which Muon's orthogonalization normalizes but SGD does not.

**Initialization**: Near-identity ($W_l = I + 0.1 \cdot \mathcal{N}(0,1)$) ensures the product $W_L \cdots W_1 \approx I + O(0.1\sqrt{L})$ at initialization, providing a well-conditioned starting point for all depths. This is critical for fair comparison -- if initialization were random, deeper networks would start with exponentially worse conditioning, confounding the LR sweep.

### Newton-Schulz Orthogonalization

This is the core of Muon's update direction computation. Given gradient matrix $G$, Newton-Schulz iteration computes the **orthogonal polar factor** $G_{\text{orth}} \approx UV^T$ where $G = U\Sigma V^T$. The iteration starts with $X_0 = G / \|G\|_F$ and applies:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^T X_k)$$

After 5 iterations, all singular values of $X$ are driven to 1, giving an orthogonal matrix. The key effect: **the update direction treats all spectral components equally**, regardless of the condition number of the gradient. This is why Muon is hypothesized to be robust to depth -- it neutralizes the spectral imbalance that makes SGD struggle in deep networks.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    """Newton-Schulz iteration for orthogonal polar factor."""
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

### Weight Initialization, Data Generation, and Forward/Backward Pass

These functions replicate the D-TEST setup exactly. The near-identity initialization ($I + 0.1\epsilon$) is critical: it ensures the initial product matrix is close to $I$, so both optimizers start from comparable conditioning across all depths. The data generation uses a fixed random target matrix and random inputs, creating a regression problem that is the same across optimizer comparisons at each seed.

In [ ]:
def init_weights(dim, depth, seed):
    """Initialize near identity for stability (same as D-TEST)."""
    rng = np.random.RandomState(seed)
    return [np.eye(dim) + rng.randn(dim, dim) * 0.1 for _ in range(depth)]

In [ ]:
def make_data(dim, seed):
    """Generate target matrix and data (same as D-TEST: single random target)."""
    rng = np.random.RandomState(seed)
    W_target = rng.randn(dim, dim) * 0.5
    X = rng.randn(dim, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    diff = pred - Y
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

### Unified Training Loop

The `train()` function handles both SGD and Muon in a single loop. The key difference:
- **SGD**: momentum update $v_t = \beta v_{t-1} + G$, then $W \leftarrow W - \text{lr} \cdot v_t$
- **Muon**: momentum update $v_t = \beta v_{t-1} + G_{\text{orth}}$, then $W \leftarrow W - \text{lr} \cdot v_t$

where $G_{\text{orth}}$ is the Newton-Schulz orthogonalized gradient. Both use the same momentum coefficient ($\beta = 0.9$), the same initialization, and the same data. The **only** difference is whether the gradient is orthogonalized before entering the momentum buffer.

A divergence guard returns `inf` for any run where loss exceeds $10^{10}$, ensuring the LR sweep gracefully handles unstable configurations.

In [ ]:
def train(weights_init, X, Y, lr, optimizer, n_steps=NUM_STEPS):
    """Train and return (final_loss, loss_history)."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    losses = []

    for step in range(n_steps):
        loss = compute_loss(weights, X, Y)
        losses.append(loss)
        if not np.isfinite(loss) or loss > 1e10:
            # Fill rest with inf
            losses.extend([float('inf')] * (n_steps - step))
            return float('inf'), losses

        grads = compute_gradients(weights, X, Y)
        for i in range(len(weights)):
            if optimizer == 'muon':
                ortho_g = newton_schulz(grads[i])
                mom[i] = MOMENTUM * mom[i] + ortho_g
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]

    final_loss = compute_loss(weights, X, Y)
    losses.append(final_loss)
    return final_loss, losses

## D-TEST's Original LR Selection for SGD (Comparison Baseline)

The original D-TEST used a physics-informed LR formula for SGD: $\text{lr} = \min\left(\frac{2}{\lambda_{\max} \cdot L}, 0.1\right)$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian proxy (computed from the product of weight singular values and data singular values). This formula has the property that it **decreases with depth** (since $\lambda_{\max}$ may grow and $L$ appears in the denominator), which is a reasonable heuristic but may systematically underestimate or overestimate the optimal LR.

We replicate this formula here to run the D-TEST protocol side-by-side with our swept-LR protocol, enabling a direct comparison of the two approaches.

In [ ]:
def dtest_sgd_lr(depth, X, Y):
    """Replicate D-TEST's lr = 2/(lambda_max * L) formula."""
    rng_state = np.random.get_state()
    np.random.seed(42)
    test_weights = init_weights(DIM, depth, 42)
    np.random.set_state(rng_state)

    W_prod = np.eye(DIM)
    for W in test_weights:
        W_prod = W @ W_prod
    sv_prod = np.linalg.svd(W_prod, compute_uv=False)
    sv_X = np.linalg.svd(X, compute_uv=False)
    N = X.shape[1]
    lambda_max = (sv_prod[0] ** 2) * (sv_X[0] ** 2) / N
    lr = min(2.0 / (lambda_max * depth), 0.1)
    return lr

### Why This Formula Matters for the Audit

The formula $\text{lr} = 2 / (\lambda_{\max} \cdot L)$ is motivated by classical optimization theory: the optimal step size for gradient descent on a quadratic with Hessian eigenvalue $\lambda_{\max}$ is $2/\lambda_{\max}$, and dividing by $L$ is a heuristic for deep networks. However:

1. **The formula may be too conservative**: Dividing by $L$ linearly may over-penalize SGD at large depths, giving it an artificially small LR while Muon's fixed LR=0.005 remains aggressive.
2. **The formula may be too aggressive**: The Hessian proxy $\lambda_{\max}$ may underestimate the true curvature, giving SGD an LR that causes slow oscillation rather than divergence.
3. **Muon's fixed LR may be accidentally good**: If Muon is inherently robust to LR choice (due to the orthogonalization normalizing gradient magnitude), then fixing LR=0.005 across depths may not be a handicap at all -- it may be near-optimal everywhere.

By sweeping LRs for both, we eliminate all three of these concerns and measure the **true** depth-dependent advantage.

## Main Experiment: Seven-Phase Confound Audit

### Execution Flow

The experiment proceeds through seven distinct phases, each building on the previous:

1. **Phase 1 -- Full LR Sweep**: For each depth $L \in \{2, 4, 8, 16\}$, sweep all LR candidates for both SGD and Muon. This is the computational core: $4 \times (9 + 8) \times 3 = 204$ training runs.

2. **Phase 2 -- Best LR Extraction**: For each (depth, optimizer) pair, select the LR with the lowest median final loss across seeds. Report the best LR and loss in a table.

3. **Phase 3 -- D-TEST Replica**: Re-run the original D-TEST protocol (formula SGD LR, fixed Muon LR=0.005) for direct comparison. This tells us what D-TEST would have measured with our setup.

4. **Phase 4 -- Linear Fit**: Fit $\log(\text{advantage})$ vs $L$ for both the swept-LR and D-TEST protocols. Compare slopes and $R^2$ values. This is the central statistical test.

5. **Phase 5 -- LR Scaling Analysis**: Examine how the optimal LR changes with depth for each optimizer. If SGD's optimal LR drops much faster than Muon's, the depth exponent may partly reflect LR sensitivity differences rather than algorithmic capability.

6. **Phase 6 -- Full LR Landscape**: Print the complete loss landscape (all LRs at all depths) to verify that the sweep found the true optimum and didn't hit boundary effects.

7. **Phase 7 -- Temporal Stability**: Measure advantage at multiple training steps (50, 100, ..., 300) to check if the depth scaling is a transient early-training phenomenon or persists through convergence.

### Final Verdict

The verdict is determined by the swept-LR $R^2$ and slope:
- $R^2 > 0.8$ and slope $> 0.03$: D-TEST SURVIVES
- $0.5 < R^2 < 0.8$: PARTIALLY SURVIVES (weakened but real)
- $R^2 < 0.5$ or slope $< 0.01$: D-TEST FAILS (retract the claim)

In [ ]:
def main():
    t_start = time.time()
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]
    total_runs = len(DEPTHS) * (len(SGD_LRS) + len(MUON_LRS)) * NUM_SEEDS
    run_count = 0

    print()
    print("=" * 110)
    print("  H6a: LR CONFOUND AUDIT OF D-TEST DEPTH SCALING")
    print("  THE PAPER'S CREDIBILITY DEPENDS ON THIS RESULT")
    print("=" * 110)
    print()
    print(f"  Setup: {DIM}x{DIM} deep linear, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"  Depths: {DEPTHS}")
    print(f"  SGD LR sweep:  {SGD_LRS}")
    print(f"  Muon LR sweep: {MUON_LRS}")
    print(f"  Total training runs: {total_runs}")
    print(f"  Seeds: {seeds}")
    print()
    print(f"  QUESTION: Does Muon's exponential depth advantage survive when we give")
    print(f"  SGD its PER-DEPTH OPTIMAL learning rate instead of a formula-based one?")
    print()

    # =========================================================================
    # PHASE 1: Full LR sweep for both optimizers at every depth
    # =========================================================================

    print("-" * 110)
    print("  PHASE 1: FULL LR SWEEP FOR BOTH OPTIMIZERS AT EVERY DEPTH")
    print("-" * 110)
    print()
    print("  Rationale: The original D-TEST used a formula lr = 2/(lambda_max * L) for SGD")
    print("  and a fixed lr = 0.005 for Muon. This asymmetry may confound the results.")
    print("  Here we sweep BOTH optimizers to find each one's true per-depth optimum.")
    print()

    # results[depth][optimizer][lr] = list of final losses across seeds
    results = {}

    for depth in DEPTHS:
        print(f"  --- Depth L={depth} ---")
        print(f"      Network: {depth} layers of {DIM}x{DIM} matrices")
        print(f"      Product matrix: {DIM}x{DIM} (W_{depth} ... W_1)")
        results[depth] = {'sgd': {}, 'muon': {}}

        # Generate data (fixed per seed, independent of depth for fair comparison)
        # But depth affects the problem structure, so we use a common data seed
        for opt_name, lr_list in [('sgd', SGD_LRS), ('muon', MUON_LRS)]:
            for lr in lr_list:
                seed_losses = []
                for s in seeds:
                    X, Y = make_data(DIM, s)
                    w_init = init_weights(DIM, depth, s + 5000)
                    final_loss, _ = train(w_init, X, Y, lr, opt_name)
                    seed_losses.append(final_loss)
                    run_count += 1

                results[depth][opt_name][lr] = seed_losses
                finite = [l for l in seed_losses if np.isfinite(l)]
                median_l = np.median(finite) if finite else float('inf')

            # Print progress
            best_lr_for_opt = None
            best_median = float('inf')
            for lr in lr_list:
                finite = [l for l in results[depth][opt_name][lr] if np.isfinite(l)]
                med = np.median(finite) if finite else float('inf')
                if med < best_median:
                    best_median = med
                    best_lr_for_opt = lr
            n_diverged = sum(1 for lr in lr_list
                             for l in results[depth][opt_name][lr]
                             if not np.isfinite(l))
            print(f"    {opt_name.upper():>5}: best_lr={best_lr_for_opt:.4f}  "
                  f"median_loss={best_median:.6e}  "
                  f"({run_count}/{total_runs} runs done)"
                  f"  [diverged: {n_diverged}/{len(lr_list)*NUM_SEEDS}]")

    elapsed = time.time() - t_start
    print(f"\n  All sweeps complete in {elapsed:.1f}s ({total_runs} runs)")
    print(f"  Average time per run: {elapsed/total_runs:.3f}s")

    # =========================================================================
    # PHASE 2: Extract best LR and loss for each (depth, optimizer)
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 2: BEST LR PER DEPTH AND OPTIMIZER")
    print("=" * 110)
    print()
    print("  This table is the core of the audit. For each depth, we show:")
    print("    - The BEST learning rate found by sweeping (not a formula)")
    print("    - The corresponding median final loss across seeds")
    print("    - The advantage ratio = SGD_loss / Muon_loss")
    print("    - log(advantage) for linear fit in Phase 4")
    print()
    print("  If the advantage grows with depth, Muon has a genuine depth-dependent edge.")
    print("  If it stays flat, D-TEST was an LR confound.")
    print()

    best = {}  # best[depth][optimizer] = {'lr': ..., 'median_loss': ..., 'mean_loss': ..., 'losses': [...]}

    for depth in DEPTHS:
        best[depth] = {}
        for opt_name, lr_list in [('sgd', SGD_LRS), ('muon', MUON_LRS)]:
            best_lr = None
            best_median = float('inf')
            for lr in lr_list:
                finite = [l for l in results[depth][opt_name][lr] if np.isfinite(l)]
                if finite:
                    med = np.median(finite)
                    if med < best_median:
                        best_median = med
                        best_lr = lr
            finite_best = [l for l in results[depth][opt_name][best_lr] if np.isfinite(l)]
            best[depth][opt_name] = {
                'lr': best_lr,
                'median_loss': best_median,
                'mean_loss': np.mean(finite_best) if finite_best else float('inf'),
                'losses': results[depth][opt_name][best_lr],
            }

    # Print the table
    print(f"  {'Depth':>5} | {'Best SGD LR':>12} {'SGD loss':>14} | "
          f"{'Best Muon LR':>12} {'Muon loss':>14} | "
          f"{'Advantage':>12} {'log(adv)':>10}")
    print(f"  {'':->5}-+-{'':->12}-{'':->14}-+-{'':->12}-{'':->14}-+-{'':->12}-{'':->10}")

    depth_arr = []
    log_advantage_arr = []
    advantage_arr = []

    for depth in DEPTHS:
        sgd_l = best[depth]['sgd']['median_loss']
        muon_l = best[depth]['muon']['median_loss']
        sgd_lr = best[depth]['sgd']['lr']
        muon_lr = best[depth]['muon']['lr']

        if muon_l > 1e-30 and np.isfinite(sgd_l) and np.isfinite(muon_l):
            advantage = sgd_l / muon_l
            log_adv = np.log(advantage)
            depth_arr.append(depth)
            log_advantage_arr.append(log_adv)
            advantage_arr.append(advantage)
        else:
            advantage = float('inf')
            log_adv = float('inf')

        print(f"  {depth:>5} | {sgd_lr:>12.4f} {sgd_l:>14.6e} | "
              f"{muon_lr:>12.4f} {muon_l:>14.6e} | "
              f"{advantage:>12.2f}x {log_adv:>10.4f}")

    # Quick sanity check on the advantage array
    print()
    if len(advantage_arr) >= 2:
        print(f"  Quick check: advantage at depth {DEPTHS[0]} = {advantage_arr[0]:.2f}x, "
              f"at depth {DEPTHS[-1]} = {advantage_arr[-1]:.2f}x")
        if advantage_arr[-1] > advantage_arr[0]:
            print(f"  --> Advantage INCREASES with depth ({advantage_arr[-1]/advantage_arr[0]:.1f}x growth)")
            print(f"      This is consistent with exponential depth scaling.")
        else:
            print(f"  --> Advantage DECREASES with depth ({advantage_arr[-1]/advantage_arr[0]:.2f}x)")
            print(f"      This suggests the depth exponent may NOT survive the audit.")
    print()

    # Also print how the best LR changes with depth for each optimizer
    print("  Observation: How does the best LR change with depth?")
    for opt_name in ['sgd', 'muon']:
        lrs_by_depth = [best[d][opt_name]['lr'] for d in DEPTHS]
        print(f"    {opt_name.upper()} best LRs: {[f'{lr:.4f}' for lr in lrs_by_depth]}")
    print()

    # =========================================================================
    # PHASE 3: D-TEST COMPARISON — original fixed-LR results
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 3: D-TEST COMPARISON (ORIGINAL FIXED-LR PROTOCOL)")
    print("=" * 110)
    print()
    print("  Replicating D-TEST's original protocol for side-by-side comparison:")
    print("    - SGD LR:  formula-based, lr = 2/(lambda_max * L)")
    print("    - Muon LR: fixed at 0.005 across all depths")
    print()
    print("  If the D-TEST replica shows a strong depth exponent but the swept-LR")
    print("  audit does not, then the D-TEST result was an LR artifact.")
    print()

    # Replicate D-TEST: fixed Muon LR=0.005, SGD LR from formula
    dtest_depth_arr = []
    dtest_log_adv_arr = []
    dtest_advantage_arr = []

    print(f"  {'Depth':>5} | {'D-TEST SGD LR':>14} {'SGD loss':>14} | "
          f"{'D-TEST Muon LR':>14} {'Muon loss':>14} | "
          f"{'Advantage':>12} {'log(adv)':>10}")
    print(f"  {'':->5}-+-{'':->14}-{'':->14}-+-{'':->14}-{'':->14}-+-{'':->12}-{'':->10}")

    DTEST_MUON_LR = 0.005

    for depth in DEPTHS:
        sgd_losses_dtest = []
        muon_losses_dtest = []
        for s in seeds:
            X, Y = make_data(DIM, s)
            w_init = init_weights(DIM, depth, s + 5000)
            sgd_lr_dtest = dtest_sgd_lr(depth, X, Y)

            # SGD run
            w_sgd = [W.copy() for W in w_init]
            fl_sgd, _ = train(w_sgd, X, Y, sgd_lr_dtest, 'sgd')
            sgd_losses_dtest.append(fl_sgd)

            # Muon run
            w_muon = [W.copy() for W in w_init]
            fl_muon, _ = train(w_muon, X, Y, DTEST_MUON_LR, 'muon')
            muon_losses_dtest.append(fl_muon)

        sgd_med = np.median([l for l in sgd_losses_dtest if np.isfinite(l)]) if any(np.isfinite(l) for l in sgd_losses_dtest) else float('inf')
        muon_med = np.median([l for l in muon_losses_dtest if np.isfinite(l)]) if any(np.isfinite(l) for l in muon_losses_dtest) else float('inf')

        if muon_med > 1e-30 and np.isfinite(sgd_med) and np.isfinite(muon_med):
            adv = sgd_med / muon_med
            log_adv = np.log(adv)
            dtest_depth_arr.append(depth)
            dtest_log_adv_arr.append(log_adv)
            dtest_advantage_arr.append(adv)
        else:
            adv = float('inf')
            log_adv = float('inf')

        sgd_lr_show = dtest_sgd_lr(depth, *make_data(DIM, seeds[0]))
        print(f"  {depth:>5} | {sgd_lr_show:>14.6f} {sgd_med:>14.6e} | "
              f"{DTEST_MUON_LR:>14.4f} {muon_med:>14.6e} | "
              f"{adv:>12.2f}x {log_adv:>10.4f}")

    # Compare D-TEST formula LRs to swept best LRs
    print()
    print("  Comparing D-TEST formula SGD LRs to swept-best SGD LRs:")
    for i, depth in enumerate(DEPTHS):
        dt_lr = dtest_sgd_lr(depth, *make_data(DIM, seeds[0]))
        sw_lr = best[depth]['sgd']['lr']
        ratio = sw_lr / dt_lr if dt_lr > 0 else float('nan')
        direction = "too LOW" if ratio > 1.5 else ("too HIGH" if ratio < 0.67 else "approximately correct")
        print(f"    Depth {depth:>2}: formula={dt_lr:.6f}, swept_best={sw_lr:.4f}, "
              f"ratio={ratio:.2f}x  --> formula is {direction}")
    print()

    # =========================================================================
    # PHASE 4: LINEAR FIT AND R^2
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 4: LINEAR FIT -- log(advantage) vs L")
    print("=" * 110)
    print()
    print("  The D-TEST claim is that log(advantage) scales linearly with depth L:")
    print("    log(advantage) = slope * L + intercept")
    print()
    print("  If true, advantage = e^(slope * L + intercept), meaning each layer")
    print("  multiplies the advantage by e^slope (the per-layer factor).")
    print()
    print("  D-TEST original: slope = 0.0953, R^2 = 0.91, per-layer factor = 1.10x")
    print()
    print("  We fit this for both the swept-LR data and the D-TEST replica data.")
    print()

    def linear_fit(depths, log_advs, label):
        """Fit log(advantage) = a*L + b, return (slope, intercept, R^2)."""
        if len(depths) < 2:
            print(f"  {label}: INSUFFICIENT DATA (only {len(depths)} points)")
            return 0.0, 0.0, 0.0

        d = np.array(depths, dtype=float)
        y = np.array(log_advs, dtype=float)
        A = np.vstack([d, np.ones(len(d))]).T
        result = np.linalg.lstsq(A, y, rcond=None)
        slope, intercept = result[0]

        ss_res = np.sum((y - (slope * d + intercept)) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1.0 - ss_res / (ss_tot + 1e-15) if ss_tot > 1e-15 else 0.0

        print(f"  {label}:")
        print(f"    Fit: log(advantage) = {slope:.4f} * L + ({intercept:.4f})")
        print(f"    Per-layer factor: e^slope = {np.exp(slope):.4f}x")
        print(f"    R^2 = {r2:.4f}")
        print(f"    Residuals: {['%.4f' % (y[i] - (slope*d[i] + intercept)) for i in range(len(d))]}")
        print()
        return slope, intercept, r2

    # Swept-LR fit (THIS IS THE AUDIT)
    slope_swept, intercept_swept, r2_swept = linear_fit(
        depth_arr, log_advantage_arr, "SWEPT-LR (per-depth best for BOTH optimizers)")

    # D-TEST-replica fit (for comparison)
    slope_dtest, intercept_dtest, r2_dtest = linear_fit(
        dtest_depth_arr, dtest_log_adv_arr, "D-TEST REPLICA (fixed Muon LR, formula SGD LR)")

    # Interpretation of the comparison
    print("  --- Phase 4 Interpretation ---")
    print(f"  D-TEST replica slope: {slope_dtest:.4f}  (per-layer: {np.exp(slope_dtest):.4f}x)")
    print(f"  Swept-LR audit slope: {slope_swept:.4f}  (per-layer: {np.exp(slope_swept):.4f}x)")
    if abs(slope_dtest) > 1e-10:
        slope_retention = slope_swept / slope_dtest * 100
        print(f"  Slope retention: {slope_retention:.1f}% of D-TEST slope survives the audit")
    print(f"  R^2 comparison: {r2_dtest:.4f} (D-TEST) vs {r2_swept:.4f} (swept)")
    print()

    # =========================================================================
    # PHASE 5: LR SCALING WITH DEPTH
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 5: HOW DOES OPTIMAL LR SCALE WITH DEPTH?")
    print("=" * 110)
    print()
    print("  This diagnostic reveals WHETHER the confound mechanism is operating.")
    print("  If SGD's optimal LR drops much faster with depth than Muon's, then")
    print("  D-TEST's fixed-LR protocol would systematically disadvantage SGD at")
    print("  greater depths, creating a spurious depth exponent.")
    print()
    print("  Conversely, if both optimizers' LRs scale similarly with depth, then")
    print("  D-TEST's protocol was fair and the depth exponent is genuine.")
    print()

    print(f"  {'Depth':>5} | {'Best SGD LR':>12} | {'Best Muon LR':>12} | "
          f"{'D-TEST SGD LR':>14} | {'SGD swept/dtest':>16}")
    print(f"  {'':->5}-+-{'':->12}-+-{'':->12}-+-{'':->14}-+-{'':->16}")

    sgd_swept_lrs = []
    muon_swept_lrs = []
    dtest_sgd_lrs = []

    for depth in DEPTHS:
        sgd_lr_swept = best[depth]['sgd']['lr']
        muon_lr_swept = best[depth]['muon']['lr']
        sgd_lr_dt = dtest_sgd_lr(depth, *make_data(DIM, seeds[0]))

        sgd_swept_lrs.append(sgd_lr_swept)
        muon_swept_lrs.append(muon_lr_swept)
        dtest_sgd_lrs.append(sgd_lr_dt)

        ratio = sgd_lr_swept / sgd_lr_dt if sgd_lr_dt > 0 else float('nan')
        print(f"  {depth:>5} | {sgd_lr_swept:>12.4f} | {muon_lr_swept:>12.4f} | "
              f"{sgd_lr_dt:>14.6f} | {ratio:>16.2f}x")

    # Check if SGD LR decreases faster with depth than Muon LR
    print()
    if len(DEPTHS) >= 2:
        sgd_lr_ratio = sgd_swept_lrs[-1] / sgd_swept_lrs[0] if sgd_swept_lrs[0] > 0 else float('nan')
        muon_lr_ratio = muon_swept_lrs[-1] / muon_swept_lrs[0] if muon_swept_lrs[0] > 0 else float('nan')
        print(f"  SGD LR ratio (depth {DEPTHS[-1]} / depth {DEPTHS[0]}):  {sgd_lr_ratio:.4f}")
        print(f"  Muon LR ratio (depth {DEPTHS[-1]} / depth {DEPTHS[0]}): {muon_lr_ratio:.4f}")
        print()
        print(f"  Interpretation:")
        print(f"    SGD best LR changes by {sgd_lr_ratio:.4f}x across the depth range")
        print(f"    Muon best LR changes by {muon_lr_ratio:.4f}x across the depth range")
        print()
        if np.isfinite(sgd_lr_ratio) and np.isfinite(muon_lr_ratio):
            if sgd_lr_ratio < muon_lr_ratio * 0.5:
                print("  WARNING: SGD's optimal LR decreases MUCH faster with depth than Muon's.")
                print("  This asymmetric scaling could EXPLAIN the depth-exponent as an LR confound.")
                print("  SGD is more LR-sensitive to depth, so a fixed LR protocol handicaps it more.")
            elif sgd_lr_ratio < muon_lr_ratio:
                print("  NOTE: SGD's optimal LR decreases somewhat faster with depth than Muon's.")
                print("  This may partially confound the depth-exponent measurement.")
            else:
                print("  GOOD: SGD and Muon optimal LRs scale similarly with depth.")
                print("  The depth-exponent is NOT primarily an LR scaling artifact.")

        # Also check if Muon's D-TEST fixed LR (0.005) was near optimal
        print()
        print(f"  Was D-TEST's fixed Muon LR (0.005) near optimal at each depth?")
        for i, depth in enumerate(DEPTHS):
            muon_optimal = muon_swept_lrs[i]
            ratio = 0.005 / muon_optimal if muon_optimal > 0 else float('nan')
            quality = "GOOD" if 0.5 <= ratio <= 2.0 else "POOR"
            print(f"    Depth {depth:>2}: optimal={muon_optimal:.4f}, "
                  f"fixed=0.005, ratio={ratio:.2f}x  [{quality}]")

    # =========================================================================
    # PHASE 6: DETAILED LR LANDSCAPE PER DEPTH
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 6: FULL LR LANDSCAPE (median final loss)")
    print("=" * 110)
    print()
    print("  This detailed landscape serves two purposes:")
    print("    1. Verify the sweep found the TRUE optimum (not at a boundary)")
    print("    2. Reveal the SHAPE of the loss-vs-LR curve at each depth")
    print()
    print("  A sharp minimum means the optimizer is LR-sensitive (bad for fixed-LR protocols).")
    print("  A flat minimum means the optimizer is LR-robust (less confound risk).")

    for depth in DEPTHS:
        print(f"\n  Depth L={depth}:")
        print(f"    SGD LRs:")
        sgd_losses_at_depth = []
        for lr in SGD_LRS:
            finite = [l for l in results[depth]['sgd'][lr] if np.isfinite(l)]
            med = np.median(finite) if finite else float('inf')
            mean = np.mean(finite) if finite else float('inf')
            marker = " <-- BEST" if lr == best[depth]['sgd']['lr'] else ""
            frac = len(finite) / NUM_SEEDS * 100
            print(f"      lr={lr:.4f}  median={med:12.6e}  mean={mean:12.6e}  "
                  f"converged={frac:.0f}%{marker}")
            if finite:
                sgd_losses_at_depth.append((lr, med))

        # Check for boundary effect
        if sgd_losses_at_depth:
            best_idx = min(range(len(sgd_losses_at_depth)), key=lambda i: sgd_losses_at_depth[i][1])
            if best_idx == 0:
                print(f"      ** WARNING: Best LR is at the LOWER boundary of the sweep **")
                print(f"         The true optimum may be at an even lower LR.")
            elif best_idx == len(sgd_losses_at_depth) - 1:
                print(f"      ** WARNING: Best LR is at the UPPER boundary of the sweep **")
                print(f"         The true optimum may be at an even higher LR.")

        print(f"    Muon LRs:")
        muon_losses_at_depth = []
        for lr in MUON_LRS:
            finite = [l for l in results[depth]['muon'][lr] if np.isfinite(l)]
            med = np.median(finite) if finite else float('inf')
            mean = np.mean(finite) if finite else float('inf')
            marker = " <-- BEST" if lr == best[depth]['muon']['lr'] else ""
            frac = len(finite) / NUM_SEEDS * 100
            print(f"      lr={lr:.4f}  median={med:12.6e}  mean={mean:12.6e}  "
                  f"converged={frac:.0f}%{marker}")
            if finite:
                muon_losses_at_depth.append((lr, med))

        # Check for boundary effect
        if muon_losses_at_depth:
            best_idx = min(range(len(muon_losses_at_depth)), key=lambda i: muon_losses_at_depth[i][1])
            if best_idx == 0:
                print(f"      ** WARNING: Best LR is at the LOWER boundary of the sweep **")
                print(f"         The true optimum may be at an even lower LR.")
            elif best_idx == len(muon_losses_at_depth) - 1:
                print(f"      ** WARNING: Best LR is at the UPPER boundary of the sweep **")
                print(f"         The true optimum may be at an even higher LR.")

    # =========================================================================
    # PHASE 7: RESIDUAL TABLE — advantage at EACH measurement point
    # =========================================================================

    print()
    print("=" * 110)
    print("  PHASE 7: ADVANTAGE AT MULTIPLE TRAINING STEPS (using best LRs)")
    print("=" * 110)
    print()
    print("  This temporal analysis checks whether the depth scaling is:")
    print("    - A transient early-training phenomenon (advantage peaks then flattens)")
    print("    - A persistent effect (advantage maintained or grows through training)")
    print("    - A late-training artifact (advantage only appears near convergence)")
    print()
    print("  If the advantage is transient, the depth exponent reflects early-phase")
    print("  dynamics (e.g., escaping saddle points) rather than fundamental scaling.")
    print()

    MEASUREMENT_STEPS = [50, 100, 150, 200, 250, 300]

    print(f"  {'Depth':>5} |", end="")
    for ms in MEASUREMENT_STEPS:
        print(f"  Step {ms:>3}", end="")
    print()
    print(f"  {'':->5}-+", end="")
    for _ in MEASUREMENT_STEPS:
        print(f"{'':->10}", end="")
    print()

    step_advantages = {}  # step_advantages[depth] = list of advantages at each step
    for depth in DEPTHS:
        # Run SGD and Muon at best LRs, store full loss curves
        sgd_curves = []
        muon_curves = []
        for s in seeds:
            X, Y = make_data(DIM, s)
            w_init = init_weights(DIM, depth, s + 5000)

            w_sgd = [W.copy() for W in w_init]
            _, losses_sgd = train(w_sgd, X, Y, best[depth]['sgd']['lr'], 'sgd')
            sgd_curves.append(losses_sgd)

            w_muon = [W.copy() for W in w_init]
            _, losses_muon = train(w_muon, X, Y, best[depth]['muon']['lr'], 'muon')
            muon_curves.append(losses_muon)

        advs_this_depth = []
        print(f"  {depth:>5} |", end="")
        for ms in MEASUREMENT_STEPS:
            sgd_at_step = [c[ms] if ms < len(c) else c[-1] for c in sgd_curves]
            muon_at_step = [c[ms] if ms < len(c) else c[-1] for c in muon_curves]
            sgd_med = np.median([l for l in sgd_at_step if np.isfinite(l)]) if any(np.isfinite(l) for l in sgd_at_step) else float('inf')
            muon_med = np.median([l for l in muon_at_step if np.isfinite(l)]) if any(np.isfinite(l) for l in muon_at_step) else float('inf')
            if muon_med > 1e-30 and np.isfinite(sgd_med):
                adv = sgd_med / muon_med
                advs_this_depth.append(adv)
                print(f"  {adv:>7.2f}x", end="")
            else:
                advs_this_depth.append(float('inf'))
                print(f"  {'INF':>8}", end="")
        print()
        step_advantages[depth] = advs_this_depth

    # Temporal stability analysis
    print()
    print("  Temporal stability analysis:")
    for depth in DEPTHS:
        advs = [a for a in step_advantages[depth] if np.isfinite(a)]
        if len(advs) >= 2:
            trend = advs[-1] / advs[0] if advs[0] > 0 else float('nan')
            variability = np.std(advs) / np.mean(advs) * 100 if np.mean(advs) > 0 else float('nan')
            label = "GROWING" if trend > 1.2 else ("SHRINKING" if trend < 0.8 else "STABLE")
            print(f"    Depth {depth:>2}: first={advs[0]:.2f}x, last={advs[-1]:.2f}x, "
                  f"trend={trend:.2f}x, CV={variability:.1f}%  [{label}]")

    # =========================================================================
    # FINAL VERDICT
    # =========================================================================

    print()
    print("=" * 110)
    print("  FINAL VERDICT: DOES D-TEST SURVIVE THE LR CONFOUND AUDIT?")
    print("=" * 110)
    print()

    print(f"  D-TEST ORIGINAL CLAIM:")
    print(f"    log(advantage) ~ {0.0953:.4f} * L, R^2 = 0.91")
    print(f"    Per-layer factor: 1.10x")
    print(f"    Interpretation: each layer gives Muon an additional 10% advantage over SGD")
    print()

    print(f"  D-TEST REPLICA (this experiment, same protocol, {NUM_SEEDS} seeds):")
    print(f"    Slope = {slope_dtest:.4f}, R^2 = {r2_dtest:.4f}")
    print(f"    Per-layer factor: {np.exp(slope_dtest):.4f}x")
    print()

    print(f"  SWEPT-LR AUDIT (per-depth best LR for BOTH optimizers):")
    print(f"    Slope = {slope_swept:.4f}, R^2 = {r2_swept:.4f}")
    print(f"    Per-layer factor: {np.exp(slope_swept):.4f}x")
    print()

    # Key comparisons
    slope_change = abs(slope_swept - slope_dtest) / (abs(slope_dtest) + 1e-15)
    r2_change = r2_dtest - r2_swept

    print(f"  COMPARISONS:")
    print(f"    Slope change: {slope_dtest:.4f} -> {slope_swept:.4f} "
          f"({slope_change*100:.1f}% change)")
    print(f"    R^2 change:   {r2_dtest:.4f} -> {r2_swept:.4f} "
          f"(dropped by {r2_change:.4f})")
    print()

    # --- VERDICT ---
    print(f"  {'='*80}")

    if r2_swept > 0.8 and slope_swept > 0.03:
        print(f"  VERDICT: D-TEST SURVIVES THE AUDIT")
        print(f"  {'='*80}")
        print()
        print(f"  The exponential depth scaling PERSISTS even when both optimizers")
        print(f"  get their per-depth optimal learning rates.")
        print(f"    - R^2 = {r2_swept:.4f} (threshold: 0.8) -- PASS")
        print(f"    - Slope = {slope_swept:.4f} (per-layer factor {np.exp(slope_swept):.4f}x)")
        if slope_change < 0.5:
            print(f"    - Slope changed by only {slope_change*100:.1f}% -- robust")
        else:
            print(f"    - Slope changed by {slope_change*100:.1f}% -- the MAGNITUDE is reduced")
            print(f"      but the QUALITATIVE finding (exponential scaling) holds.")
        print()
        print(f"  The depth-exponent is a REAL algorithmic advantage, not an LR confound.")
        print(f"  Muon's Newton-Schulz orthogonalization provides a genuine per-layer")
        print(f"  benefit that compounds with network depth. This is consistent with the")
        print(f"  RG gauge-fixing interpretation: each layer introduces spectral redundancy")
        print(f"  that Muon's orthogonalization removes, while SGD must contend with it.")

    elif r2_swept > 0.5 and slope_swept > 0.01:
        print(f"  VERDICT: D-TEST PARTIALLY SURVIVES -- WEAKENED BUT NOT DEAD")
        print(f"  {'='*80}")
        print()
        print(f"  The exponential trend exists but is weaker after LR correction.")
        print(f"    - R^2 = {r2_swept:.4f} (below 0.8 threshold but above 0.5)")
        print(f"    - Slope = {slope_swept:.4f} (per-layer factor {np.exp(slope_swept):.4f}x)")
        print()
        if slope_change > 0.5:
            print(f"  WARNING: The original D-TEST OVERSTATED the effect by {slope_change*100:.0f}%")
            print(f"  due to LR confounding. The paper should report the corrected values.")
        else:
            print(f"  The effect size is similar but noisier with proper LR tuning.")
        print()
        print(f"  The depth-dependent advantage is real but the quantitative estimate was")
        print(f"  inflated by D-TEST's asymmetric LR selection. The corrected per-layer")
        print(f"  factor of {np.exp(slope_swept):.4f}x (vs original 1.10x) should be used.")

    elif r2_swept < 0.5 or slope_swept < 0.01:
        print(f"  VERDICT: D-TEST DOES NOT SURVIVE -- RETRACT THE CLAIM")
        print(f"  {'='*80}")
        print()
        print(f"  The exponential depth scaling DISAPPEARS when both optimizers")
        print(f"  get proper per-depth LR tuning.")
        print(f"    - R^2 dropped from {r2_dtest:.4f} to {r2_swept:.4f}")
        if slope_swept < 0.01:
            print(f"    - Slope dropped from {slope_dtest:.4f} to {slope_swept:.4f}")
            print(f"      The per-layer advantage factor is essentially 1.0x (no scaling)")
        print()
        print(f"  The D-TEST result was an LR CONFOUND. The apparent exponential scaling")
        print(f"  came from SGD being given increasingly suboptimal LRs at greater depths")
        print(f"  while Muon's fixed LR happened to be near-optimal across depths.")
        print()
        print(f"  This does NOT mean Muon has no advantage -- it may still be faster per-step")
        print(f"  or more robust to hyperparameters. But the specific claim of EXPONENTIAL")
        print(f"  depth scaling is not supported when both optimizers are properly tuned.")

    else:
        print(f"  VERDICT: INCONCLUSIVE")
        print(f"  {'='*80}")
        print()
        print(f"  Cannot determine if the depth scaling is real or artifactual.")
        print(f"  R^2 = {r2_swept:.4f}, slope = {slope_swept:.4f}")
        print(f"  More seeds or a wider depth range may be needed to resolve this.")

    print()
    elapsed_total = time.time() - t_start
    print(f"  Total experiment time: {elapsed_total:.1f}s")
    print(f"  Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    print("=" * 110)
    print("  EXPERIMENT COMPLETE")
    print("=" * 110)
    print()

## Execute the Audit

Running the full seven-phase confound audit. The output below will report:

- **Phase 1**: Full LR sweep progress for each depth and optimizer (204 training runs total)
- **Phase 2**: Best LR and loss table with advantage ratios -- the primary data for the audit
- **Phase 3**: D-TEST replica results using the original protocol, for side-by-side comparison
- **Phase 4**: Linear fits of $\log(\text{advantage})$ vs depth for both protocols, with $R^2$ and residuals
- **Phase 5**: LR scaling analysis -- how each optimizer's optimal LR changes with depth
- **Phase 6**: Complete LR landscape at each depth with boundary warnings
- **Phase 7**: Temporal stability -- advantage measured at 6 training checkpoints to check persistence
- **Final Verdict**: Pass/fail determination with scientific interpretation

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Scientific Implications

### What This Experiment Tests

H6a is the most important credibility check in the experimental program. The D-TEST claimed that Muon's advantage over SGD grows exponentially with depth (1.10x per layer, $R^2 = 0.91$), which would be strong evidence for the RG gauge-fixing interpretation. But H6 already showed that a related claim (130x curvature advantage) was entirely an LR artifact. H6a asks: **is the depth exponent also an artifact?**

The audit works by replacing D-TEST's asymmetric LR protocol (formula for SGD, fixed for Muon) with a symmetric one (sweep for both). If the depth exponent survives, it is genuine. If it collapses, it was a confound.

### Interpreting the Three Possible Outcomes

**If D-TEST SURVIVES ($R^2 > 0.8$, slope $> 0.03$):**
- The exponential depth scaling is real and robust to LR tuning.
- Muon's Newton-Schulz orthogonalization provides a genuine compounding benefit per layer.
- This strongly supports the RG gauge-fixing interpretation: each layer introduces spectral redundancy (gauge degrees of freedom) that Muon removes but SGD does not.
- The per-layer factor gives a quantitative prediction: at depth $L$, Muon should converge $\sim e^{\text{slope} \cdot L}$ times faster than SGD, even when both are optimally tuned.

**If D-TEST PARTIALLY SURVIVES ($0.5 < R^2 < 0.8$):**
- There is a real depth-dependent advantage, but it is weaker or noisier than D-TEST claimed.
- The original per-layer factor (1.10x) was likely inflated by the LR confound.
- The corrected per-layer factor should replace the original in all claims.
- The RG interpretation is weakened but not falsified -- the depth scaling exists but with larger error bars.

**If D-TEST FAILS ($R^2 < 0.5$ or slope $< 0.01$):**
- The exponential depth scaling was an LR artifact, similar to the 130x curvature artifact in H6.
- The D-TEST claim should be retracted or heavily qualified.
- This does NOT mean Muon has no advantage at all -- it may still be faster per-step or more robust to hyperparameters. But the specific claim of exponential compounding with depth is not supported.
- The RG gauge-fixing interpretation needs to find other evidence for its depth-scaling prediction.

### Implications for Future Work

Regardless of the outcome, H6a establishes an important methodological principle: **all optimizer comparisons must use per-depth LR tuning**. Any claim of depth-dependent advantage that uses a fixed or formula-based LR for one optimizer but not the other is suspect.

If the depth exponent does survive, the natural follow-up experiments are:
- **H6b**: Test whether the surviving advantage is in the *direction* of Muon's update (spectral equalization quality) rather than its magnitude
- **Wider depth range**: Test at depths 32, 64 to see if the exponential trend continues or saturates
- **Nonlinear networks**: Test whether the depth exponent persists in networks with ReLU activations, where the optimization landscape is fundamentally different

If the depth exponent does not survive, the key question becomes: what IS Muon's genuine advantage, and how does it scale? The answer may involve LR robustness, per-step convergence rate, or saddle point escape -- none of which require depth-dependent scaling.